In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# **Step 1 — Check Kaggle Environment**

In [2]:
# Check Python version

import sys

print("Python version:", sys.version)

Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


# **Step 2 — Install the libraries**

In [3]:
!pip install -q transformers accelerate

# **Step 3 — Check Transformers**

In [4]:
import transformers

print("Transformers version:", transformers.__version__)

Transformers version: 5.0.0


# **Step 4 — Check GPU Availability**

In [5]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.10.0+cpu
GPU available: False


# **Step 5 — Load a Conversational Model**

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

print("Loading model:", MODEL_NAME)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,  # float32 becuase CPU is using
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print("Model loaded successfully on:", device)

Loading model: Qwen/Qwen2.5-0.5B-Instruct


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully on: cpu


# **Step 6 — Build the Chatbot Function**

In [7]:
# System prompt — you can change and check chatbot "personality" or can set business use-case
SYSTEM_PROMPT = "You are a helpful, friendly AI assistant for a business. Answer clearly and concisely."

# Conversation history here
conversation_history = [
    {"role": "system", "content": SYSTEM_PROMPT}
]

def chat(user_message, max_new_tokens=200):
    """Ek user message le kar, model se reply generate karta hai aur history update karta hai."""
    conversation_history.append({"role": "user", "content": user_message})

    # Chat template apply and make prompt
    prompt = tokenizer.apply_chat_template(
        conversation_history,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

    # only new generated part
    new_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]
    reply = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    conversation_history.append({"role": "assistant", "content": reply})
    return reply

# **Step 7 — Test the Chatbot**

In [8]:
response = chat("Hello! Who are you and what can you help me with?")
print("Bot:", response)

Bot: I am an AI designed to assist users with various tasks and information needs across different industries. I'm here to provide guidance on topics like general knowledge, language translation, customer support, education, entertainment, and more. If there's anything specific you'd like help with or need assistance with, feel free to ask!


In [9]:
response = chat("Can you give me 3 tips for improving customer service in a small business?")
print("Bot:", response)

Bot: Certainly! Here are three tips that can significantly improve your customer service:

1. **Personalization**: Always greet customers with a smile and personalize interactions by addressing them by name. This not only makes the interaction personal but also shows empathy towards their needs.

2. **Promptness and Efficiency**: Treat all inquiries equally and handle requests quickly. Ensure that any issues are resolved promptly without delay. This shows that you value your customers' time and are proactive in solving problems.

3. **Feedback and Continuous Improvement**: Regularly gather feedback from customers through surveys or online reviews. Use this feedback to continuously improve your services. It’s important to listen to what customers have to say and make improvements based on their experiences.

By focusing on these aspects of customer service, you can enhance the overall experience for both customers and businesses alike.


# **Step 8 — Interactive Chat Loop**

In [10]:
print("Chatbot ready! Type 'exit' to stop.\n")

while True:
    user_input = input("You: ")
    if user_input.strip().lower() in ["exit", "quit"]:
        print("Bot: Goodbye!")
        break
    reply = chat(user_input)
    print("Bot:", reply)

Chatbot ready! Type 'exit' to stop.



You:  hello


Bot: Hello! How can I assist you today?


You:  i want plan my daily rotine.


Bot: Of course! Planning a daily routine is a great way to organize your day effectively. Here’s a simple guide to help you get started:

### Morning Routine

1. **Wake Up**: Start your day with a good night's sleep.
2. **Dress**: Put on comfortable clothes that fit well.
3. **Morning Exercise (Optional)**: Consider doing some light exercise such as stretching, yoga, or a walk outside if possible.
4. **Breakfast (If Available)**: Have a healthy breakfast ready before work or school.
5. **Hydration**: Drink water throughout the day to stay hydrated.
6. **Mindfulness or Meditation**: Spend a few minutes meditating or practicing mindfulness during the morning hours.

### Central Routine

1. **Check Emails/Notifications**: Open emails and messages that require immediate attention.
2. **Set Priorities**: Identify tasks that are most urgent or important and tackle those first.
3. **Take Breaks**: Take short breaks between tasks to


You:  exit


Bot: Goodbye!


# **Next Steps — Making This Business-Ready**

# **Step 9 — Load a Bigger Model (Optional)**

In [21]:
# If you want better answers, you can switch to a bigger model.
# This works best if you have a GPU session enabled in Kaggle.
def load_model(model_name):
    # This function replaces the current model with a new one
    global tokenizer, model, device, MODEL_NAME

    MODEL_NAME = model_name
    print("Loading model:", MODEL_NAME)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    print("Model loaded on:", device)

# Example: try a bigger model (needs GPU)
# load_model("Qwen/Qwen2.5-1.5B-Instruct")

# **Step 10 — Simple Knowledge Base Search (Basic RAG)**

In [12]:
# A small list of business facts (add your own here)
KNOWLEDGE_BASE = [
    "Our office hours are Monday to Saturday, 9 AM to 6 PM.",
    "Products can be returned within 7 days with a receipt.",
    "Delivery usually takes 3 to 5 working days.",
    "Customer support is available on WhatsApp and email.",
]

def find_relevant_facts(user_message):
    # Very simple search: check if any word from the fact
    # also appears in the user's message
    message_words = set(user_message.lower().split())
    matches = []

    for fact in KNOWLEDGE_BASE:
        fact_words = set(fact.lower().split())
        if message_words & fact_words:  # if there is any common word
            matches.append(fact)

    return matches

def chat_with_context(user_message, max_new_tokens=200):
    # Find any relevant facts first
    relevant_facts = find_relevant_facts(user_message)

    if relevant_facts:
        facts_text = "\n".join(f"- {f}" for f in relevant_facts)
        full_message = f"Company info:\n{facts_text}\n\nQuestion: {user_message}"
    else:
        full_message = user_message

    return chat(full_message, max_new_tokens=max_new_tokens)

In [13]:
response = chat_with_context("What are your office hours?")
print("Bot:", response)

Bot: Your office hours are from Monday to Saturday, 9 AM to 6 PM.


# **Step 11 — Simple Web Interface with Gradio**

In [14]:
!pip install -q gradio

In [15]:
import gradio as gr

def gradio_chat_fn(message, history):
    # Gradio passes the message here, we just reuse our existing chat function
    return chat_with_context(message)

demo = gr.ChatInterface(
    fn=gradio_chat_fn,
    title="Business AI Assistant",
    description="Ask me anything about the business!",
)

demo.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://1c0cc19a0330250c42.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# **Step 12 — Basic Guardrails**

In [16]:
# Add words you want to block here
BLOCKED_WORDS = ["hack", "bomb", "weapon", "illegal"]

FALLBACK_MESSAGE = "Sorry, I can only help with business-related questions. How else can I assist you?"

def is_blocked(message):
    # Check if any blocked word appears in the message
    lowered = message.lower()
    for word in BLOCKED_WORDS:
        if word in lowered:
            return True
    return False

def safe_chat(user_message, max_new_tokens=200):
    # Check the message before sending it to the model
    if is_blocked(user_message):
        return FALLBACK_MESSAGE
    return chat_with_context(user_message, max_new_tokens=max_new_tokens)

In [17]:
print(safe_chat("Can you give me tips for improving customer service?"))
print(safe_chat("How do I hack into a system?"))

Absolutely! Improving customer service involves several key steps:

1. **Be Friendly and Professional**: Your first impression is crucial. Be courteous and respectful in all interactions.

2. **Greet Customers Promptly**: Addressing customers when they arrive is essential. A warm welcome can go a long way.

3. **Use Clear Communication**: Clearly communicate expectations and follow up after transactions. Make sure to use clear language and avoid jargon unless it is understood.

4. **Handle Complaints Promptly**: Listen actively to complaints and address them promptly. Provide solutions where appropriate.

5. **Follow Up**: After resolving a complaint, ensure that you follow up to check if the issue has been resolved. Follow-up helps build trust and transparency.

6. **Offer Solutions**: When customers reach out for solutions, be prepared to offer advice or suggestions. Flexibility is key!

7. **Encourage Feedback**: Encourage customers to share their thoughts and concerns with you. Res

# **Step 13 — Persistent Memory with SQLite**

In [26]:
import sqlite3
import json

DB_PATH = "chat_history.db"

def init_db():
    # Creates the table if it doesn't already exist
    conn = sqlite3.connect(DB_PATH)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS conversations (
            session_id TEXT PRIMARY KEY,
            history TEXT
        )
    """)
    conn.commit()
    conn.close()

init_db()

In [25]:
def save_history(session_id, history):
    # Converts the history list into text (JSON) and saves it under session_id
    conn = sqlite3.connect(DB_PATH)
    conn.execute(
        "INSERT OR REPLACE INTO conversations (session_id, history) VALUES (?, ?)",
        (session_id, json.dumps(history))
    )
    conn.commit()
    conn.close()

In [24]:
def load_history(session_id):
    # Loads history for a session_id, or starts a fresh one if none exists
    conn = sqlite3.connect(DB_PATH)
    row = conn.execute(
        "SELECT history FROM conversations WHERE session_id = ?",
        (session_id,)
    ).fetchone()
    conn.close()

    if row:
        return json.loads(row[0])
    else:
        return [{"role": "system", "content": SYSTEM_PROMPT}]

In [23]:
# Load (or start) a conversation for this user
conversation_history = load_history("user_123")

# Chat normally — this updates conversation_history in memory
response = chat("Hi, what services do you offer?")
print("Bot:", response)

# Save it to the database
save_history("user_123", conversation_history)

Bot: As an AI language model, I don't offer direct physical services like businesses or companies. However, I'm here to assist with a wide range of tasks including but not limited to:

1. Language translation: Can understand, interpret, translate between multiple languages.

2. Chatbot support: Answer questions, provide information, handle inquiries, and manage conversations using various models.

3. Natural language processing (NLP): Process human language data in order to extract useful insights from it.

4. Virtual assistants: Offer assistance with everyday tasks such as scheduling appointments, answering questions, and managing files.

5. Content creation: Generate text based on prompts or articles written by humans.

6. Personalization: Improve user experiences through personalized recommendations and content generation.

7. Customer service: Provide help in resolving customer queries, providing support, and handling complaints.

8. Data analysis: Analyze large amounts of data to 

In [22]:
print(load_history("user_123"))

[{'role': 'system', 'content': 'You are a helpful, friendly AI assistant for a business. Answer clearly and concisely.'}]


In [27]:
print(load_history("user_123"))

[{'role': 'system', 'content': 'You are a helpful, friendly AI assistant for a business. Answer clearly and concisely.'}, {'role': 'user', 'content': 'Hi, what services do you offer?'}, {'role': 'assistant', 'content': "As an AI language model, I don't offer direct physical services like businesses or companies. However, I'm here to assist with a wide range of tasks including but not limited to:\n\n1. Language translation: Can understand, interpret, translate between multiple languages.\n\n2. Chatbot support: Answer questions, provide information, handle inquiries, and manage conversations using various models.\n\n3. Natural language processing (NLP): Process human language data in order to extract useful insights from it.\n\n4. Virtual assistants: Offer assistance with everyday tasks such as scheduling appointments, answering questions, and managing files.\n\n5. Content creation: Generate text based on prompts or articles written by humans.\n\n6. Personalization: Improve user experien

In [28]:
print(load_history("user_123"))

[{'role': 'system', 'content': 'You are a helpful, friendly AI assistant for a business. Answer clearly and concisely.'}, {'role': 'user', 'content': 'Hi, what services do you offer?'}, {'role': 'assistant', 'content': "As an AI language model, I don't offer direct physical services like businesses or companies. However, I'm here to assist with a wide range of tasks including but not limited to:\n\n1. Language translation: Can understand, interpret, translate between multiple languages.\n\n2. Chatbot support: Answer questions, provide information, handle inquiries, and manage conversations using various models.\n\n3. Natural language processing (NLP): Process human language data in order to extract useful insights from it.\n\n4. Virtual assistants: Offer assistance with everyday tasks such as scheduling appointments, answering questions, and managing files.\n\n5. Content creation: Generate text based on prompts or articles written by humans.\n\n6. Personalization: Improve user experien

In [29]:
print(load_history("user_123"))

[{'role': 'system', 'content': 'You are a helpful, friendly AI assistant for a business. Answer clearly and concisely.'}, {'role': 'user', 'content': 'Hi, what services do you offer?'}, {'role': 'assistant', 'content': "As an AI language model, I don't offer direct physical services like businesses or companies. However, I'm here to assist with a wide range of tasks including but not limited to:\n\n1. Language translation: Can understand, interpret, translate between multiple languages.\n\n2. Chatbot support: Answer questions, provide information, handle inquiries, and manage conversations using various models.\n\n3. Natural language processing (NLP): Process human language data in order to extract useful insights from it.\n\n4. Virtual assistants: Offer assistance with everyday tasks such as scheduling appointments, answering questions, and managing files.\n\n5. Content creation: Generate text based on prompts or articles written by humans.\n\n6. Personalization: Improve user experien